## Import

In [1]:
import joblib

In [2]:
from ML.first_method.first_method import first_method
from physics.Iso_data_handler import Iso_data_handler
from physics.Data_visualiser import Data_visualiser
from ML.utils.Data_preparator import Data_preparator
from ML.utils.Model_evaluator import Model_evaluator

In [3]:
pre_path = "../../../../../../"
physical_model = "MIST"
path_to_data = pre_path + "data/MIST_v1.2_vvcrit0.0_basic_isos/"
path_to_final_models_A = 'model/final_models/Model_A_partial/MIST/'
path_to_final_models_B = 'model/final_models/Model_B/MIST/'

## Data preparation

In [4]:
iso_handler = Iso_data_handler(path_to_data, 
                              ['log10_isochrone_age_yr', 'log_Teff', 'log_g', 'phase', 'metallicity', 'star_mass', 'log_R'], 
                              physical_model, reclassify=True)

iso_df = iso_handler.get_isochrone_dataframe()

Reading MIST dataframe from MIST_reclassified_iso_full_data.csv file...


In [5]:
phase_filtered_iso_df = Data_preparator.filter_data(iso_df, {'phase':[0, 2, 3], 'star_mass':('<', 14)})

In [6]:
X_train, X_ivs, y_train, y_ivs, categories_train, categories_ivs = \
    Data_preparator.split_data(phase_filtered_iso_df, x_cols=['log10_isochrone_age_yr', 'log_Teff', 'log_g', 'metallicity'], 
                               y_cols=['star_mass', 'log_R'], categories_cols=['phase'], random_state=12, print_stats=True)

print(X_train.shape, X_ivs.shape)
print(y_train.shape, y_ivs.shape)
print(categories_train.shape, categories_ivs.shape)

Training set statistics:
Range in train data for the star_mass parameter : 0.0999979840073621 - 13.998624815474695
Median value in train data for the star_mass parameter: 2.0464342686349397
Mean value in train data for the star_mass parameter: 3.3973846370701644

Range in train data for the log_R parameter : -0.9974747647513328 - 3.082068881511837
Median value in train data for the log_R parameter: 0.5381900688843242
Mean value in train data for the log_R parameter: 0.6069651248403705

Testing set statistics:
Range in test data for the star_mass parameter : 0.0999981896729906 - 13.99946885003588
Median value in test data for the star_mass parameter: 2.048860774010381
Mean value in test data for the star_mass parameter: 3.399812923045874

Range in test data for the log_R parameter : -0.9974234436680278 - 3.0827540579576227
Median value in test data for the log_R parameter: 0.5398730026691951
Mean value in test data for the log_R parameter: 0.6057854356870224

(389468, 4) (129823, 4)
(38

## Test

In [7]:
print(X_train)

[[ 9.1         3.57541153  5.16538436 -1.25      ]
 [ 8.15        4.1504679   3.62604644 -3.5       ]
 [ 8.45        3.61701139  1.65730142  0.25      ]
 ...
 [ 7.65        4.37994054  4.04647909 -2.        ]
 [ 9.1         3.77433584  2.46084104 -1.75      ]
 [ 9.75        3.64477221  2.26972281  0.25      ]]


In [8]:
print(X_train[0])
print(y_train[0])

[ 9.1         3.57541153  5.16538436 -1.25      ]
[ 0.16795298 -0.75314325]


In [10]:
index = 0
age = X_train[index][0]
log_teff_1 = X_train[index][1]
log_g_1 = X_train[index][2]
metallicity = X_train[index][3]

mass_1 = y_train[index][0]
radius_1 = y_train[index][1]

star_mass1, log_R1, log_Teff2, log_g2, log_R2 = first_method(age, metallicity, log_teff_1, log_g_1, 1, 
                                                             pre_path + path_to_final_models_A + "KNN_compress_0.pkl",
                                                             pre_path + path_to_final_models_B + "KNN_compress_0.pkl")

print(f"Known parameters : log_age : {age}, metallicity : {metallicity}, log_Teff : {log_teff_1}, log_g : {log_g_1}")
print(f"Predicted parameters : star_mass : {star_mass1}, log_R : {log_R1}")
print(f"Expected parameters : star_mass : {mass_1}, log_R : {radius_1}")

print()

KNeighborsRegressor(algorithm='kd_tree', leaf_size=100, n_jobs=10, p=1.0,
                    weights='distance')
Known parameters : log_age : 9.1, metallicity : -1.25, log_Teff : 3.575411527945205, log_g : 5.16538435806818
Predicted parameters : star_mass : 0.6592885811890726, log_R : 0.07984384721918932
Expected parameters : star_mass : 0.1679529774043952, log_R : -0.7531432450219406

